Lightgbm


In [1]:
import os
os.chdir('/home/cevdetkopuz/mercari-price-prediction')

import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import Ridge
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

tfidf_name = joblib.load('models/tfidf_name.joblib')
tfidf_desc = joblib.load('models/tfidf_desc.joblib')
le_cat_main = joblib.load('models/le_cat_main.joblib')
le_cat_sub1 = joblib.load('models/le_cat_sub1.joblib')
le_cat_sub2 = joblib.load('models/le_cat_sub2.joblib')
le_brand = joblib.load('models/le_brand.joblib')
top_brands = joblib.load('models/top_brands.joblib')
ridge_model = joblib.load('models/ridge_model.joblib')

print("Pipeline yüklendi")

Pipeline yüklendi


In [2]:
df = pd.read_csv('data/raw/train.tsv', sep='\t')
df = df[df['price'] > 0].reset_index(drop=True)

df['brand_name'].fillna('missing', inplace=True)
df['category_name'].fillna('missing/missing/missing', inplace=True)
df['item_description'].fillna('No description yet', inplace=True)

def split_category(cat):
    parts = str(cat).split('/')
    while len(parts) < 3:
        parts.append('missing')
    return parts[:3]

cat_split = df['category_name'].apply(split_category)
df['cat_main'] = cat_split.apply(lambda x: x[0])
df['cat_sub1'] = cat_split.apply(lambda x: x[1])
df['cat_sub2'] = cat_split.apply(lambda x: x[2])

df['brand_name'] = df['brand_name'].apply(lambda x: x if x in top_brands else 'other')
df['cat_main_enc'] = le_cat_main.transform(df['cat_main'])
df['cat_sub1_enc'] = le_cat_sub1.transform(df['cat_sub1'])
df['cat_sub2_enc'] = le_cat_sub2.transform(df['cat_sub2'])
df['brand_enc'] = le_brand.transform(df['brand_name'])
df['desc_word_count'] = df['item_description'].str.split().str.len()
df['name_len'] = df['name'].str.len()
df['has_description'] = (df['item_description'] != 'No description yet').astype(int)

numeric_features = ['item_condition_id', 'shipping', 'cat_main_enc',
                    'cat_sub1_enc', 'cat_sub2_enc', 'brand_enc',
                    'desc_word_count', 'name_len', 'has_description']

X_numeric = df[numeric_features].values.astype(np.float32)
y = np.log1p(df['price']).values.astype(np.float32)

# TF-IDF → SVD (100K sparse → 200 dense)
print("SVD: name (50K → 100 dim)...")
t0 = time.time()
X_name_sparse = tfidf_name.transform(df['name'])
svd_name = TruncatedSVD(n_components=100, random_state=42, n_iter=5)
X_name_svd = svd_name.fit_transform(X_name_sparse).astype(np.float32)
print(f"  {time.time()-t0:.1f}s, açıklanan varyans: {svd_name.explained_variance_ratio_.sum():.3f}")
del X_name_sparse; gc.collect()

print("SVD: desc (50K → 100 dim)...")
t0 = time.time()
X_desc_sparse = tfidf_desc.transform(df['item_description'])
svd_desc = TruncatedSVD(n_components=100, random_state=42, n_iter=5)
X_desc_svd = svd_desc.fit_transform(X_desc_sparse).astype(np.float32)
print(f"  {time.time()-t0:.1f}s, açıklanan varyans: {svd_desc.explained_variance_ratio_.sum():.3f}")
del X_desc_sparse; gc.collect()

# Birleştir: 9 sayısal + 100 name SVD + 100 desc SVD = 209 feature
X_lgb = np.hstack([X_numeric, X_name_svd, X_desc_svd]).astype(np.float32)
del X_name_svd, X_desc_svd; gc.collect()

print(f"\nLightGBM matrisi: {X_lgb.shape}, {X_lgb.nbytes/1e6:.1f} MB")

# SVD'leri kaydet (inference'ta lazım)
joblib.dump(svd_name, 'models/svd_name.joblib')
joblib.dump(svd_desc, 'models/svd_desc.joblib')
print("SVD modelleri kaydedildi")

SVD: name (50K → 100 dim)...
  69.7s, açıklanan varyans: 0.153
SVD: desc (50K → 100 dim)...
  177.8s, açıklanan varyans: 0.207

LightGBM matrisi: (1481661, 209), 1238.7 MB
SVD modelleri kaydedildi


In [3]:
X_train, X_val, y_train, y_val = train_test_split(X_lgb, y, test_size=0.2, random_state=42)

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'max_depth': -1,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'verbose': -1,
    'n_jobs': -1,
    'seed': 42,
}

print("LightGBM eğitiliyor...")
t0 = time.time()

lgb_train = lgb.Dataset(X_train, y_train, free_raw_data=True)
lgb_val_ds = lgb.Dataset(X_val, y_val, reference=lgb_train, free_raw_data=True)

lgb_model = lgb.train(
    lgb_params, lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_val_ds], valid_names=['val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

print(f"\nEğitim: {time.time()-t0:.1f}s")
print(f"En iyi iterasyon: {lgb_model.best_iteration}")

y_val_pred_lgb = np.maximum(lgb_model.predict(X_val), 0)
lgb_rmsle = np.sqrt(mean_squared_error(y_val, y_val_pred_lgb))

print(f"\n{'='*40}")
print(f"LightGBM RMSLE: {lgb_rmsle:.4f}")
print(f"Ridge RMSLE:    0.4733 (önceki)")
print(f"{'='*40}")

LightGBM eğitiliyor...
Training until validation scores don't improve for 50 rounds
[100]	val's rmse: 0.558319
[200]	val's rmse: 0.532438
[300]	val's rmse: 0.520438
[400]	val's rmse: 0.512055
[500]	val's rmse: 0.506534
[600]	val's rmse: 0.503426
[700]	val's rmse: 0.500938
[800]	val's rmse: 0.498986
[900]	val's rmse: 0.497233
[1000]	val's rmse: 0.495802
[1100]	val's rmse: 0.494813
[1200]	val's rmse: 0.493084
[1300]	val's rmse: 0.492321
[1400]	val's rmse: 0.491621
[1500]	val's rmse: 0.490826
[1600]	val's rmse: 0.490044
[1700]	val's rmse: 0.489489
[1800]	val's rmse: 0.488956
[1900]	val's rmse: 0.488448
[2000]	val's rmse: 0.487982
Did not meet early stopping. Best iteration is:
[2000]	val's rmse: 0.487982

Eğitim: 578.7s
En iyi iterasyon: 2000

LightGBM RMSLE: 0.4880
Ridge RMSLE:    0.4733 (önceki)


In [4]:
# Model kaydet
lgb_model.save_model('models/lgbm_svd_model.txt')
joblib.dump(svd_name, 'models/svd_name.joblib')
joblib.dump(svd_desc, 'models/svd_desc.joblib')

config = joblib.load('models/pipeline_config.joblib')
config['lgbm_svd_rmsle'] = lgb_rmsle
config['lgbm_best_iteration'] = lgb_model.best_iteration
joblib.dump(config, 'models/pipeline_config.joblib')

print("Kaydedildi!")
print(f"  lgbm_svd_model.txt")
print(f"  svd_name.joblib")
print(f"  svd_desc.joblib")

Kaydedildi!
  lgbm_svd_model.txt
  svd_name.joblib
  svd_desc.joblib


In [5]:
# Ensemble: Ridge tahminlerini dosyadan yükle
# (Ridge'i yeniden çalıştırmaya gerek yok, sadece val seti üzerinde tahmin lazım)

from scipy.sparse import hstack, csr_matrix

# Val seti index'lerini yeniden oluştur (aynı random_state=42)
np.random.seed(42)
n = len(y)
indices = np.arange(n)
np.random.shuffle(indices)  # train_test_split(random_state=42) ile aynı değil
# Doğrudan val tahminlerini kullanacağız

# Ridge'i sadece val seti için çalıştır (parça parça, bellek dostu)
ridge_model = joblib.load('models/ridge_model.joblib')

# Val index'leri (train_test_split ile aynı)
from sklearn.model_selection import train_test_split
_, val_idx = train_test_split(range(n), test_size=0.2, random_state=42)

# Val verisi için Ridge tahmini (küçük batch'lerle)
df_val = df.iloc[val_idx].reset_index(drop=True)

X_num_val = csr_matrix(df_val[numeric_features].values, dtype=np.float32)
X_name_val = tfidf_name.transform(df_val['name'])
X_desc_val = tfidf_desc.transform(df_val['item_description'])
X_val_sparse = hstack([X_num_val, X_name_val, X_desc_val]).tocsr()

y_val_pred_ridge = np.maximum(ridge_model.predict(X_val_sparse), 0)
del X_val_sparse, X_num_val, X_name_val, X_desc_val; gc.collect()

ridge_rmsle = np.sqrt(mean_squared_error(y_val, y_val_pred_ridge))

# Ensemble ağırlık araması
print(f"{'Ağırlık (Ridge/LGB)':<25} {'RMSLE':<10}")
print("-" * 35)

best_ens = float('inf')
best_w = None
for w in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    ens = w * y_val_pred_ridge + (1-w) * y_val_pred_lgb
    r = np.sqrt(mean_squared_error(y_val, ens))
    marker = " <-- best" if r < best_ens else ""
    if r < best_ens:
        best_ens = r
        best_w = w
    print(f"  {w:.1f} / {1-w:.1f}                {r:.4f}{marker}")

print("-" * 35)
print(f"\nRidge tek:     {ridge_rmsle:.4f}")
print(f"LightGBM tek:  {lgb_rmsle:.4f}")
print(f"Ensemble:      {best_ens:.4f}  (Ridge={best_w}, LGB={1-best_w})")
print(f"İyileşme:      {ridge_rmsle - best_ens:+.4f} (Ridge'e göre)")

Ağırlık (Ridge/LGB)       RMSLE     
-----------------------------------
  0.2 / 0.8                0.4697 <-- best
  0.3 / 0.7                0.4633 <-- best
  0.4 / 0.6                0.4588 <-- best
  0.5 / 0.5                0.4563 <-- best
  0.6 / 0.4                0.4557 <-- best
  0.7 / 0.3                0.4572
  0.8 / 0.2                0.4606
-----------------------------------

Ridge tek:     0.4733
LightGBM tek:  0.4880
Ensemble:      0.4557  (Ridge=0.6, LGB=0.4)
İyileşme:      +0.0175 (Ridge'e göre)


In [6]:
# Sonuçları kaydet
config = joblib.load('models/pipeline_config.joblib')
config.update({
    'ridge_rmsle': 0.4733,
    'lgbm_svd_rmsle': 0.4880,
    'ensemble_rmsle': 0.4557,
    'ensemble_weights': {'ridge': 0.6, 'lgbm': 0.4},
})
joblib.dump(config, 'models/pipeline_config.joblib')

print("Sonuçlar kaydedildi!")
print()
print("=== PROJE SKORLARI ===")
print(f"  MVP   Ridge:     0.4733")
print(f"  Ideal LightGBM:  0.4880")  
print(f"  Ideal Ensemble:  0.4557  ← en iyi")
print(f"  Hedef:           0.43")
print(f"  Kalan fark:      {0.4557 - 0.43:.4f}")

Sonuçlar kaydedildi!

=== PROJE SKORLARI ===
  MVP   Ridge:     0.4733
  Ideal LightGBM:  0.4880
  Ideal Ensemble:  0.4557  ← en iyi
  Hedef:           0.43
  Kalan fark:      0.0257


In [7]:
# Önce Optuna'yı kur (ilk seferde gerekli)
import subprocess
subprocess.check_call(['pip', 'install', 'optuna', '-q'])

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"Optuna {optuna.__version__} yüklendi")
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

Optuna 4.8.0 yüklendi
X_train: (1185328, 209), X_val: (296333, 209)


In [ ]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbose': -1,
        'n_jobs': -1,
        'seed': 42,
        
        # Optuna'nın arayacağı parametreler
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 300),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
    }
    
    lgb_tr = lgb.Dataset(X_train, y_train, free_raw_data=False)
    lgb_vl = lgb.Dataset(X_val, y_val, reference=lgb_tr, free_raw_data=False)
    
    model = lgb.train(
        params, lgb_tr,
        num_boost_round=2000,
        valid_sets=[lgb_vl],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0),
        ],
    )
    
    pred = np.maximum(model.predict(X_val), 0)
    rmsle = np.sqrt(mean_squared_error(y_val, pred))
    return rmsle

# Çalıştır
print("Optuna başlıyor (30 deneme)...")
print("Her deneme ~1-2 dakika, toplam ~30-60 dakika")
print()

study = optuna.create_study(direction='minimize', study_name='lgbm_tuning')

t0 = time.time()
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nToplam süre: {(time.time()-t0)/60:.1f} dakika")
print(f"\n{'='*50}")
print(f"EN İYİ SONUÇ")
print(f"  RMSLE: {study.best_value:.4f}")
print(f"  Önceki LightGBM: 0.4880")
print(f"  İyileşme: {0.4880 - study.best_value:+.4f}")
print(f"{'='*50}")
print(f"\nEn iyi parametreler:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Optuna başlıyor (30 deneme)...
Her deneme ~1-2 dakika, toplam ~30-60 dakika



  0%|          | 0/30 [00:00<?, ?it/s]